# 01 · Ingestión y transformación del dataset (EA1)

**Proyecto:** Análisis Descriptivo de la Estancia Hospitalaria y Factores Operativos en Admisiones Médicas.

**Objetivo:** obtener el dataset completo desde Hugging Face con una clase (`HFDatasetLoader`), aplicar transformaciones documentadas, generar un resumen `.txt` con otra clase (`DatasetSummarizer`) y exportar CSV de apoyo.

Flujo: **Hugging Face → DataFrame original → transformaciones → DataFrame transformado → `reports/data-summary.txt` + CSV (`data/raw`, `data/processed`)**.

Dataset: [`Nicolybgs/healthcare_data`](https://huggingface.co/datasets/Nicolybgs/healthcare_data). Ejecute las celdas en orden.

> **Importante:** el DataFrame en memoria contiene **todas** las filas. El límite de 100.000 filas se aplica *únicamente* a los CSV exportados (primeras filas, sin aleatoriedad); por tanto, los CSV son una **parte** del dataset, no el dataset completo.

## 1. Importar bibliotecas y clases

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))  # permite `import src...` desde cualquier carpeta

import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

from src import config
from src.data.dataset_io import make_loader, export_csv_head
from src.data import transformations as tr
from src.utils.reporter import DatasetSummarizer

print("Raíz del proyecto:", ROOT.name)

D:\Documents\Study\IUDigital\proyecto_integrado_3\healthcare-hospital-stay-analytics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raíz del proyecto: healthcare-hospital-stay-analytics


## 2. Carga del dataset completo

`available_splits()` consulta Hugging Face para verificar configuración y split (no se asumen). `load()` descarga **todas** las filas y las devuelve como `DataFrame`. No se necesita token: el dataset es público.

In [2]:
loader = make_loader()   # HFDatasetLoader(dataset_id, config, split); sin muestreo
print("Configuraciones y splits disponibles:", loader.available_splits())
raw = loader.load()
info = loader.info()
info

Configuraciones y splits disponibles: {'default': ['train']}


{'dataset_id': 'Nicolybgs/healthcare_data',
 'config': 'default',
 'split': 'train',
 'filas_cargadas': 500000,
 'columnas': ['Available Extra Rooms in Hospital',
  'Department',
  'Ward_Facility_Code',
  'doctor_name',
  'staff_available',
  'patientid',
  'Age',
  'gender',
  'Type of Admission',
  'Severity of Illness',
  'health_conditions',
  'Visitors with Patient',
  'Insurance',
  'Admission_Deposit',
  'Stay (in days)'],
 'muestreo': 'dataset completo (sin muestreo)'}

## 3. Registros originales

In [3]:
raw.head()

,Available Extra Rooms in Hospital,Department,Ward_Facility_Code,doctor_name,staff_available,patientid,Age,gender,Type of Admission,Severity of Illness,health_conditions,Visitors with Patient,Insurance,Admission_Deposit,Stay (in days)
0,4,gynecology,D,Dr Sophia,0,33070,41-50,Female,Trauma,Extreme,Diabetes,4,Yes,2966.408696,8
1,4,gynecology,B,Dr Sophia,2,34808,31-40,Female,Trauma,Minor,Heart disease,2,No,3554.835677,9
2,2,gynecology,B,Dr Sophia,8,44577,21-30,Female,Trauma,Extreme,Diabetes,2,Yes,5624.733654,7
3,4,gynecology,D,Dr Olivia,7,3695,31-40,Female,Urgent,Moderate,None,4,No,4814.149231,8
4,2,anesthesia,E,Dr Mark,10,108956,71-80,Male,Trauma,Moderate,Diabetes,2,No,5169.269637,34


## 4. Inspección inicial: dimensiones, columnas y tipos

In [4]:
print(f"Dimensiones: {raw.shape[0]:,} filas x {raw.shape[1]} columnas")
pd.DataFrame({"tipo": raw.dtypes.astype(str), "nulos": raw.isna().sum(), "únicos": raw.nunique()})

Dimensiones: 500,000 filas x 15 columnas


,tipo,nulos,únicos
Available Extra Rooms in Hospital,int64,0,18
Department,object,0,5
Ward_Facility_Code,object,0,6
doctor_name,object,0,9
staff_available,int64,0,11
patientid,int64,0,126399
Age,object,0,10
gender,object,0,3
Type of Admission,object,0,3
Severity of Illness,object,0,3


## 5. Transformaciones

Se aplican **seis** transformaciones (funciones de `src/data/transformations.py`, reutilizadas por el notebook 02). Cada una devuelve un DataFrame nuevo y un registro con las filas antes/después. Si una operación no encuentra casos por corregir, el resultado lo muestra tal cual.

### 5.1 Quitar espacios al inicio y al final en columnas de texto
Evita que `"Trauma "` y `"Trauma"` cuenten como categorías distintas.

In [5]:
t1, log1 = tr.strip_text_values(raw)
log1

{'paso': 'strip_text_values',
 'descripcion': 'Eliminación de espacios iniciales/finales en columnas de texto.',
 'filas_antes': 500000,
 'filas_despues': 500000,
 'columnas_antes': 15,
 'columnas_despues': 15,
 'valores_modificados': 0,
 'por_columna': {}}

### 5.2 Conversión validada a tipos numéricos
`pd.to_numeric(errors="coerce")` detecta valores no convertibles; se **reportan** (no se eliminan). También se reportan valores negativos detectados (p. ej. en `patientid`), que se conservan porque la ficha no explica su significado.

In [6]:
t2, log2 = tr.convert_numeric_types(t1)
log2

{'paso': 'convert_numeric_types',
 'descripcion': 'Conversión validada a tipos numéricos (pd.to_numeric); valores no convertibles se reportan, no se eliminan.',
 'filas_antes': 500000,
 'filas_despues': 500000,
 'columnas_antes': 15,
 'columnas_despues': 15,
 'valores_no_convertibles': 0,
 'no_convertibles_por_columna': {},
 'valores_negativos_detectados': {'patientid': 888}}

### 5.3 Detección de registros exactamente duplicados
Se eliminan solo filas idénticas en **todas** las columnas. `patientid` no es único en este dataset, por lo que no se usa para deduplicar.

In [7]:
print("patientid repetidos:", int(t2["patientid"].duplicated().sum()), "(un mismo id aparece en varias filas)")
t3, log3 = tr.drop_exact_duplicates(t2)
log3

patientid repetidos: 373601 (un mismo id aparece en varias filas)


{'paso': 'drop_exact_duplicates',
 'descripcion': 'Eliminación de registros exactamente duplicados (todas las columnas iguales).',
 'filas_antes': 500000,
 'filas_despues': 500000,
 'columnas_antes': 15,
 'columnas_despues': 15,
 'duplicados_eliminados': 0}

### 5.4 Normalización de nombres de columnas a `snake_case`
Se conserva la correspondencia entre nombre original y nuevo.

In [8]:
t4, log4 = tr.normalize_column_names(t3)
pd.DataFrame({"original": list(log4["correspondencia"]), "normalizado": list(log4["correspondencia"].values())})

,original,normalizado
0,Available Extra Rooms in Hospital,available_extra_rooms_in_hospital
1,Department,department
2,Ward_Facility_Code,ward_facility_code
3,doctor_name,doctor_name
4,staff_available,staff_available
5,patientid,patientid
6,Age,age
7,gender,gender
8,Type of Admission,type_of_admission
9,Severity of Illness,severity_of_illness


### 5.5 Estandarización de categorías
Colapsa espacios internos y convierte el texto a tipo `category`. `severity_of_illness` (Minor < Moderate < Extreme) y `age` (rangos) quedan como categorías **ordenadas**, sin convertirlas a números arbitrarios. No se recodifican valores (se conserva `Asthama` tal como viene en la fuente) y los nulos de `health_conditions` **no se imputan**: la ficha no documenta su significado.

In [9]:
t5, log5 = tr.standardize_categories(t4)
print({k: v for k, v in log5.items() if k in ("valores_modificados", "severidad_fuera_de_catalogo")})
display(t5.dtypes.astype(str).rename("tipo").to_frame().T)
t5["severity_of_illness"].dtype

{'valores_modificados': 0, 'severidad_fuera_de_catalogo': 0}


,available_extra_rooms_in_hospital,department,ward_facility_code,doctor_name,staff_available,patientid,age,gender,type_of_admission,severity_of_illness,health_conditions,visitors_with_patient,insurance,admission_deposit,stay_in_days
tipo,int64,category,category,category,int64,int64,category,category,category,category,category,int64,category,float64,int64


CategoricalDtype(categories=['Minor', 'Moderate', 'Extreme'], ordered=True, categories_dtype=object)

### 5.6 Enriquecimiento: grupo de disponibilidad de personal
`staff_available` (0–10) se agrupa en `Bajo (0-3)`, `Medio (4-6)` y `Alto (7+)` para poder comparar grupos. Los umbrales son una **convención del proyecto**, no de la ficha.

In [10]:
t6, log6 = tr.add_staff_availability_group(t5)
log6["distribucion"]

{'Bajo (0-3)': 180291, 'Medio (4-6)': 137125, 'Alto (7+)': 182584}

### 5.7 Verificación de equivalencia
`apply_all()` (usada por el notebook 02) debe producir exactamente el mismo DataFrame que los pasos manuales.

In [11]:
processed, log = tr.apply_all(raw)
assert processed.equals(t6), "apply_all() debe coincidir con los pasos manuales"
pd.DataFrame(log)[["paso", "filas_antes", "filas_despues", "columnas_antes", "columnas_despues"]]

,paso,filas_antes,filas_despues,columnas_antes,columnas_despues
0,strip_text_values,500000,500000,15,15
1,convert_numeric_types,500000,500000,15,15
2,drop_exact_duplicates,500000,500000,15,15
3,normalize_column_names,500000,500000,15,15
4,standardize_categories,500000,500000,15,15
5,add_staff_availability_group,500000,500000,15,16


## 6. Comparación entre datos originales y transformados

In [12]:
pd.DataFrame({
    "original": [raw.shape[0], raw.shape[1], int(raw.isna().sum().sum()), int(raw.duplicated().sum())],
    "transformado": [processed.shape[0], processed.shape[1], int(processed.isna().sum().sum()), int(processed.duplicated().sum())],
}, index=["filas", "columnas", "nulos totales", "filas duplicadas"])

,original,transformado
filas,500000,500000
columnas,15,16
nulos totales,151888,151888
filas duplicadas,0,0


## 7. Verificaciones de calidad

In [13]:
assert len(processed) == len(raw) - log[2]["duplicados_eliminados"], "Conteo de filas inconsistente"
assert processed.shape[1] == raw.shape[1] + 1, "Debe haber una columna nueva (grupo de personal)"
assert processed["staff_availability_group"].notna().all(), "Todas las filas deben tener grupo de personal"
assert set(processed["severity_of_illness"].dropna().unique()) <= set(tr.SEVERITY_ORDER)
assert processed["stay_in_days"].min() > 0 and processed["admission_deposit"].min() > 0
print("Verificaciones superadas.")
print("Nulos por columna (solo las que tienen):")
nulos = processed.isna().sum()
nulos[nulos > 0].to_frame("nulos")

Verificaciones superadas.
Nulos por columna (solo las que tienen):


,nulos
health_conditions,151888


## 8. Exportación limitada de CSV
Se exportan como máximo `CSV_MAX_ROWS` (100.000) filas, **las primeras en orden**, para no superar el peso permitido por GitHub (aviso a 50 MB). El DataFrame completo no se modifica.

In [14]:
exports = [export_csv_head(raw, config.RAW_CSV), export_csv_head(processed, config.PROCESSED_CSV)]
display(pd.DataFrame(exports))
for e in exports:
    n = len(pd.read_csv(ROOT / e["archivo"]))
    assert n == e["filas_exportadas"] <= config.CSV_MAX_ROWS
    print(f"{e['archivo']}: {n:,} filas leídas de vuelta; {'PARCIAL' if e['es_parcial'] else 'completo'}; {e['tamano_mb']} MB")

,archivo,filas_dataset_completo,filas_exportadas,es_parcial,tamano_mb
0,data/raw/healthcare_raw.csv,500000,100000,True,9.10
1,data/processed/healthcare_processed.csv,500000,100000,True,10.19


data/raw/healthcare_raw.csv: 100,000 filas leídas de vuelta; PARCIAL; 9.1 MB
data/processed/healthcare_processed.csv: 100,000 filas leídas de vuelta; PARCIAL; 10.19 MB


## 9. Resumen del DataFrame transformado completo (`DatasetSummarizer`)

In [15]:
ruta = DatasetSummarizer().save(
    processed,
    config.DATA_SUMMARY_TXT,
    dataset_info=info,
    transformations=tr.describe_log(log),
    exports=exports,
)
texto = ruta.read_text(encoding="utf-8")
assert f"Filas: {len(processed)}" in texto and f"Columnas: {processed.shape[1]}" in texto
print("Generado:", ruta.relative_to(ROOT).as_posix(), f"({ruta.stat().st_size} bytes)")
print(texto)

Generado: reports/data-summary.txt (4573 bytes)
RESUMEN DEL DATAFRAME
Fecha de generación: 2026-10-25 17:09:38 UTC-0500

Dataset: Nicolybgs/healthcare_data
Configuración: default
Split: train
Carga: dataset completo (sin muestreo)

Filas: 500000
Columnas: 16
Filas duplicadas: 0

Nombres de columnas:
available_extra_rooms_in_hospital, department, ward_facility_code, doctor_name, staff_available, patientid, age, gender, type_of_admission, severity_of_illness, health_conditions, visitors_with_patient, insurance, admission_deposit, stay_in_days, staff_availability_group

Tipos de datos:
- available_extra_rooms_in_hospital: int64
- department: category
- ward_facility_code: category
- doctor_name: category
- staff_available: int64
- patientid: int64
- age: category
- gender: category
- type_of_admission: category
- severity_of_illness: category
- health_conditions: category
- visitors_with_patient: int64
- insurance: category
- admission_deposit: float64
- stay_in_days: int64
- staff_availa

## 10. Conclusiones

* Se cargó el dataset completo desde Hugging Face y se transformó **sin eliminar registros**: los controles de calidad muestran que no había duplicados exactos ni espacios sobrantes que corregir.
* Se normalizaron nombres y tipos, se definieron categorías ordenadas (severidad y edad) y se creó `staff_availability_group`.
* `health_conditions` conserva sus nulos porque la ficha no explica su significado.
* `reports/data-summary.txt` describe el DataFrame transformado **completo**; los CSV de `data/` contienen solo las primeras 100.000 filas.